In [1]:
import requests
import pandas as pd
from shapely.geometry import shape, Point
from rtree import index
from tqdm import tqdm

In [9]:
query = """select *, st_x(the_geom) as lng, st_y(the_geom) as lat,
case when st_y(the_geom)<42 then 1 else 0 end as locatable

from car_ped_stops where districtoccur in ('06','09') and extract(year from datetimeoccur) = 2019
"""
response = requests.get(f"https://phl.carto.com/api/v2/sql?q={query}")
df = pd.DataFrame(response.json()['rows'])

In [21]:
import sqlite3

In [22]:
df = pd.read_sql("select * from car_ped_stops_quarterly where year=2019",con=sqlite3.connect("data/open_data_philly_2024_08_01.db"))

In [23]:
df

,districtoccur,psa,quarter,Race,Gender,Age Range,year,n_stopped,n_people_in_stopped_vehicles,n_searched,n_arrested,n_contraband,n_frisked,n_intruded,quarter_dt_str,quarter_dt,quarter_date,q_str
0,01,1,2019-Q1,All Other Races,Female,25-34,2019,1,1,0,0,0,0,0,2019-01-01T00:00:00.000000Z,2019-01-01 00:00:00+00:00,2019-01-01,Q1
1,01,1,2019-Q1,All Other Races,Female,35-44,2019,2,2,0,0,0,0,0,2019-01-01T00:00:00.000000Z,2019-01-01 00:00:00+00:00,2019-01-01,Q1
2,01,1,2019-Q1,All Other Races,Female,45-54,2019,2,2,0,0,0,0,0,2019-01-01T00:00:00.000000Z,2019-01-01 00:00:00+00:00,2019-01-01,Q1
3,01,1,2019-Q1,All Other Races,Female,Under 25,2019,2,2,0,0,0,0,0,2019-01-01T00:00:00.000000Z,2019-01-01 00:00:00+00:00,2019-01-01,Q1
4,01,1,2019-Q1,All Other Races,Male,25-34,2019,4,4,0,0,0,0,0,2019-01-01T00:00:00.000000Z,2019-01-01 00:00:00+00:00,2019-01-01,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12236,77,0,2019-Q4,White,Male,25-34,2019,11,11,10,0,0,0,10,2019-10-01T00:00:00.000000Z,2019-10-01 00:00:00+00:00,2019-10-01,Q4
12237,77,0,2019-Q4,White,Male,35-44,2019,12,13,11,0,0,0,11,2019-10-01T00:00:00.000000Z,2019-10-01 00:00:00+00:00,2019-10-01,Q4
12238,77,0,2019-Q4,White,Male,45-54,2019,13,13,11,0,0,0,11,2019-10-01T00:00:00.000000Z,2019-10-01 00:00:00+00:00,2019-10-01,Q4
12239,77,0,2019-Q4,White,Male,55-64,2019,10,12,10,0,0,0,10,2019-10-01T00:00:00.000000Z,2019-10-01 00:00:00+00:00,2019-10-01,Q4


In [12]:
# 97.3% have valid locations

with open('maps/police_psas.geojson', 'r') as f:
    geojson_data = json.load(f)

idx = index.Index()
polygons = []
for pos, feature in enumerate(geojson_data['features']):
    polygon = shape(feature['geometry'])
    polygons.append((polygon, feature['properties']))  # Store the polygon and its properties
    idx.insert(pos, polygon.bounds)  # Insert polygon bounds into the spatial index

# Function to find the feature containing a point
def find_feature(lat, lng, geojson_data):
    point = Point(lng, lat)
    matches = list(idx.intersection(point.bounds))
    for match in matches:
        polygon, properties = polygons[match]
        if polygon.contains(point):
            return {"new_psa": properties['PSA_NUM'][-1],
                    "new_districtoccur": properties['PSA_NUM'][:2]
                   }
    return None

def find_feature2(lat, lng, geojson_data):
    point = Point(lng, lat)
    for feature in [x for x in geojson_data['features'] if x['properties']['DISTRICT__'] in ['06','09']]:
        polygon = shape(feature['geometry'])
        if polygon.contains(point):
            return {"new_psa": feature['properties']['PSA_NUM'][-1],
                    "new_districtoccur": feature['properties']['DISTRICT__']
                   }
    return None

outputs = []

for row in tqdm([x for x in df.itertuples()]):
    outputs.append(find_feature2(row.lat, row.lng, psa_new))

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18295/18295 [00:02<00:00, 6408.50it/s]


In [13]:
find_feature(39.962326,-75.161444,psa_new)


{'new_psa': '1', 'new_districtoccur': '09'}

In [14]:
find_feature(39.9508907,-75.1820714,psa_new)

{'new_psa': '3', 'new_districtoccur': '09'}

In [17]:
import folium
import json
from shapely.geometry import shape, GeometryCollection

# Load GeoJSON data from a file
with open('maps/police_psas_old.geojson', 'r') as f:
    geojson_data_old = json.load(f)

# Extract geometries from GeoJSON features
geometries = [shape(feature['geometry']) for feature in geojson_data_old['features']]
# Combine geometries into a single GeometryCollection
combined = GeometryCollection(geometries)
# Calculate the centroid
centroid = combined.centroid
center = [centroid.y, centroid.x]


# Create a Folium map centered around some coordinates (latitude, longitude)
m = folium.Map(location=center, zoom_start=12)

# Add GeoJSON layer to the map
folium.GeoJson(geojson_data_old, 
                style_function=lambda feature: {
        'fillColor': '#ffaf00',
        'weight': 4,
        'dashArray': '5, 5',
        'fillOpacity': .2, 'opacity': 1
    },
               tooltip=folium.GeoJsonTooltip(fields=['PSA_NUM'], localize=True)).add_to(m)

folium.GeoJson(geojson_data,
    style_function=lambda feature: {
        'fillColor': '#ffaf00',
        'color': '#000000',
        'weight': 4,
        'dashArray': '5, 5',
        'fillOpacity': 0,'opacity': 0.3
    },
               tooltip=folium.GeoJsonTooltip(fields=['PSA_NUM'], localize=True)
              ).add_to(m)

m

In [18]:
# 6-1 -> 9-2


import folium
import json
from shapely.geometry import shape, GeometryCollection

# Extract geometries from GeoJSON features
geometries = [shape(feature['geometry']) for feature in geojson_data_old['features']]
# Combine geometries into a single GeometryCollection
combined = GeometryCollection(geometries)
# Calculate the centroid
centroid = combined.centroid
center = [centroid.y, centroid.x]


# Create a Folium map centered around some coordinates (latitude, longitude)
m = folium.Map(location=center, zoom_start=12)

# Add GeoJSON layer to the map
folium.GeoJson(geojson_data, 
                style_function=lambda feature: {
        'fillColor': '#ffaf00',
        'weight': 4,
        'dashArray': '5, 5',
        'fillOpacity': .2, 'opacity': 1
    },
               tooltip=folium.GeoJsonTooltip(fields=['PSA_NUM'], localize=True)).add_to(m)

m